# EVX pack_risk_eol: model selection on Cloudera AI

The nightly pipeline trains three candidates, picks the highest out-of-fold PR-AUC, runs the KPI gate,
registers the winner and deploys `evx-uc01-service`. This notebook runs **the same functions** as
`usecases/uc01_pack_risk/train.py` and `mlcore/gate.py`, step by step on the published champion metrics.

| Step | What it mirrors | What happens here |
|---|---|---|
| 1 | Champion metrics | load `metrics.json` or the joblib the register job uploaded |
| 2 | Candidate comparison | bar chart of out-of-fold PR-AUC (selection metric) and AUROC (gate only) |
| 3 | Gate (dry run) | `decide()` against `config/lakehouse.json` — PASS on the platform champion |
| 4 | Rejection demo | a stricter AUROC floor in memory only; config is not edited |
| 5 | Production read-back | decision counts from Impala `gold.uc01_pack_risk_score` |
| 6 | Deployed endpoint | one call to `evx-uc01-service` when CAI keys are set |

**Why this matters:** the notebook is not a separate prototype. It imports the same gate and reads the same
artefacts as `ci/register_deploy.py`, so there is no "works in the notebook, rewrite for production" gap.

The data is **synthetic** history scored on 2026-11-05. Say so up front.

Full selection record: [README_CAI.md](../README_CAI.md). Consumption layer: the **EVX Quality and
Reliability Workbench** Streamlit app on subdomain `evx-workbench`.

> **Say:** "Same gate, same schema hash, same action NONE. The only difference in this session is that I am
> not retraining unless you ask — I am replaying what shipped."

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from common import load_env, lakehouse_config
from mlcore.gate import decide

load_env()
CFG = lakehouse_config()
GATE = CFG["model"]["gate"]
BASELINE_PATH = ROOT / "usecases" / "uc01_pack_risk" / "baseline.json"
METRICS_PATH = ROOT / "outputs" / "uc01" / "pack_risk_eol" / "metrics.json"
CHAMPION_PATH = ROOT / "models" / "champion" / "model.joblib"
DEMO_VIN = "XEVS1K26052000026"
DEMO_PACK = "PKPL12605200026"
pd.set_option("display.max_colwidth", 72)
print(f"repo: {ROOT.name} | baseline: {BASELINE_PATH.exists()} | champion joblib: {CHAMPION_PATH.exists()}")

## Step 1. Load the champion metrics

**What:** the metrics dict written by `train.py` after the five batch-grouped folds.

**Why:** selection happens on out-of-fold PR-AUC before anything is registered. These numbers are what
`decide()` and `baseline.json` refer to.

**How:** prefer `models/champion/model.joblib` on CAI (what the register job uploaded); fall back to
`outputs/uc01/pack_risk_eol/metrics.json` on a laptop.

In [ ]:
def load_metrics() -> dict:
    if CHAMPION_PATH.exists():
        import joblib
        art = joblib.load(CHAMPION_PATH)
        return art["metrics"]
    if METRICS_PATH.exists():
        return json.loads(METRICS_PATH.read_text())
    raise FileNotFoundError("run train locally or upload models/champion/model.joblib to CAI")

metrics = load_metrics()
baseline = json.loads(BASELINE_PATH.read_text()) if BASELINE_PATH.exists() else None
summary = {k: metrics[k] for k in ("winner", "oof_pr_auc", "oof_auroc", "oof_pr_lift", "oof_ece",
                                   "train_positives", "test_positives", "fold_positives_min", "mixed_share",
                                   "worklist_threshold", "abstain_threshold")}
pd.Series(summary, name="value").round(4)

## Step 2. Three candidates — PR-AUC selects, AUROC gates

**What:** LightGBM, logistic regression and histogram gradient boosting, each scored inside grouped folds.

**Why:** logistic wins on this synthetic factory because the planted effects are nearly linear. Gradient
boosters are not retuned to overturn that.

**How:** `winner = max(candidates, key=oof_pr_auc)`. TEST packs are never used here.

In [ ]:
cands = pd.DataFrame(metrics["candidates"]).T
cands["selected"] = cands.index == metrics["winner"]
display(cands.round(4))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
order = cands.sort_values("oof_pr_auc", ascending=False)
colors = ["#2ca02c" if n == metrics["winner"] else "#1f77b4" for n in order.index]
order["oof_pr_auc"].plot.bar(ax=ax[0], color=colors)
ax[0].set_title("Out-of-fold PR-AUC (selection)")
ax[0].set_ylabel("PR-AUC")
order["oof_auroc"].plot.bar(ax=ax[1], color=colors)
ax[1].axhline(GATE["min_oof_auroc"], ls="--", c="gray", label="gate floor")
ax[1].set_title("Out-of-fold AUROC (gate only)")
ax[1].legend()
plt.tight_layout()
plt.show()

## Step 3. The KPI gate (dry run)

**What:** `mlcore.gate.decide` against `config/lakehouse.json` `model.gate`, plus non-regression vs
`baseline.json`.

**Why:** a failed gate still writes `model.joblib` locally but sets `deployable=false`. Register and deploy
refuse the file.

**How:** empty error list means PASS. Salt must be `platform` for deployability (checked at register time).

In [ ]:
errors = decide(metrics, GATE, baseline)
print("GATE", "PASS — deployable on platform salt" if not errors else "FAIL")
for e in errors:
    print("GATE_FAIL", e)
if not errors:
    checks = [
        ("train positives", metrics["train_positives"], f">= {GATE['min_positives_per_split']}"),
        ("test positives", metrics["test_positives"], f">= {GATE['min_positives_per_split']}"),
        ("smallest fold", metrics["fold_positives_min"], f">= {GATE['min_positives_per_fold']}"),
        ("oof AUROC", round(metrics["oof_auroc"], 3), f">= {GATE['min_oof_auroc']}"),
        ("PR-AUC lift", round(metrics["oof_pr_lift"], 2), f">= {GATE['min_pr_auc_lift']}"),
        ("ECE", round(metrics["oof_ece"], 4), f"<= {GATE['max_ece']}"),
        ("mixed packs", round(metrics["mixed_share"], 3), f"<= {GATE['mixed_pack_max']}"),
    ]
    pd.DataFrame(checks, columns=["check", "value", "limit"])

**What a rejection looks like.** The same metrics against a stricter gate (AUROC floor 0.90), built in memory
only — `config/lakehouse.json` is not touched. Run it: people trust a gate more once they have seen it say no.

In [ ]:
strict = {**GATE, "min_oof_auroc": 0.90}
reject = decide(metrics, strict, baseline)
print("GATE", "PASS" if not reject else "REJECTED the candidate")
pd.DataFrame({"reason": reject})

## In production: scored vehicles in Impala

**What:** the published `gold.uc01_pack_risk_score` table from CDE job `rsingh-evx-uc01-score`.

**Why:** the lakehouse score and the CAI endpoint must agree on decision, score, schema hash and `action`
NONE.

**How:** needs Impala credentials in the environment (`EVX_IMPALA_*`). On a laptop: `set -a; source .env; set +a`.

In [ ]:
prefix = CFG["db_prefix"]
gold = f"{prefix}_gold"
try:
    sys.path.insert(0, str(ROOT / "scripts"))
    import run_semantic as sem
    eng = sem.Impala(prefix)
    counts = eng.rows(
        f"SELECT decision, count(*) AS n FROM {gold}.uc01_pack_risk_score "
        f"WHERE _business_date = (SELECT max(_business_date) FROM {gold}.uc01_pack_risk_score) "
        f"GROUP BY decision ORDER BY n DESC")
    df = pd.DataFrame(counts, columns=["decision", "vehicles"])
    display(df)
    df.plot.bar(x="decision", y="vehicles", legend=False, title="Latest score date — decisions", figsize=(7, 3))
    plt.tight_layout()
    plt.show()
    none_action = eng.rows(f"SELECT count(*) FROM {gold}.uc01_pack_risk_score WHERE action <> 'NONE'")[0][0]
    print(f"rows with action other than NONE: {none_action}")
except Exception as e:
    print(f"Impala not available ({type(e).__name__}): open the workbench or run on CAI with EVX_IMPALA_* set")

### The deployed model: `evx-uc01-service`

One predict call for a known vehicle (needs `EVX_CAI_API_KEY` and the model `access_key` in the project
environment as `EVX_MODEL_ACCESS_KEY`). The body uses the factory feature columns from
`usecases/uc01_pack_risk/build_features.py`.

If keys are not set, use the **Test** tab on the model deployment in CAI instead.

In [ ]:
import requests
from usecases.uc01_pack_risk.build_features import FEATURE_COLUMNS

host = os.environ.get("EVX_CAI_HOST", "").rstrip("/")
user_key = os.environ.get("EVX_CAI_API_KEY")
access_key = os.environ.get("EVX_MODEL_ACCESS_KEY")
if host and user_key and access_key:
    svc = host.replace("https://", "")
    if not svc.startswith("modelservice."):
        svc = f"modelservice.{svc}"
    url = f"https://{svc}/model"
    # Minimal feature payload: zeros are enough for a smoke decision on the demo VIN shape.
    payload = {
        "accessKey": access_key,
        "request": {
            "vin": DEMO_VIN,
            "pack_serial": DEMO_PACK,
            "safety_floor": 0,
            "features": {c: 0 for c in FEATURE_COLUMNS},
        },
    }
    r = requests.post(url, json=payload, headers={"Authorization": f"Bearer {user_key}"}, timeout=60)
    print("HTTP", r.status_code)
    if r.ok:
        body = r.json()
        show = {k: body.get(k) for k in ("decision", "reason", "score", "action", "schema_hash")}
        print(json.dumps(show, indent=2))
    else:
        print(r.text[:300])
else:
    print("Set EVX_CAI_HOST, EVX_CAI_API_KEY and EVX_MODEL_ACCESS_KEY to call the endpoint from here.")
    print("Workbench URL: https://evx-workbench.<your-cai-host>/ (SSO login)")

## Likely questions

- **"Why logistic, not LightGBM?"** Highest out-of-fold PR-AUC on factory features. AUROC is reported and
  gated, not the selection metric.
- **"Can CI deploy?"** No. Salt `ci` prints `GATE_RESULT SKIPPED` or sets `deployable=false`. Only `platform`
  salt passes `register_deploy.py`.
- **"Why so many ABSTAIN rows?"** Isotonic calibration piles scores on a plateau inside the abstain band.
  Thresholds were not retuned to force a 10% worklist on the full population.
- **"Does the model take action?"** No. `action` is always `NONE`. Safety floor and worklist are for human
  prioritisation only.
- **"Is this synthetic?"** Yes. Be upfront — the README and workbench brand come from runtime config, not
  from this repo.
- **"Where do I click?"** CAI Experiments `evx-uc01`, Registered Models `evx-uc01-pack-risk-eol`, Deployments
  `evx-uc01-service`, Application **EVX Quality and Reliability Workbench**.